# POC — Evaluating the Knowledge Assistant

In notebook `11` you **built** a Company Knowledge Assistant (RAG + tools). The obvious next question (notebook `08`) is:

> **How do we know it actually works — with evidence, not vibes?**

This notebook makes every concept from `08.evaluation` runnable by **grading the assistant we built**. Think of it as writing *unit tests for an AI system*.

### What each part teaches

| Part | Concept (notebook 08) | What you build |
|------|-----------------------|----------------|
| 1 | Ground truth & eval dataset | A small labelled test set |
| 2 | Exact / deterministic checks | Keyword/assertion-style scoring |
| 3 | Precision / Recall / F1 | Measure **retrieval** quality |
| 4 | Relevance (LLM-as-a-judge) | Does the answer address the question? |
| 5 | Faithfulness (groundedness) | Is the answer supported by the context? |
| 6 | Non-determinism | Why scores wobble + how to handle it |
| 7 | Eval pipeline + latency/cost | One run that scores everything |
| 8 | A/B comparison | Prove a change is better (top_k=1 vs 3) |

### The golden rule (keep it in mind throughout)

```
Change ONE thing (prompt OR model OR chunk size OR k)
        |
        v
   re-run evals
        |
        v
   scores up?  -> keep it      scores down? -> roll back
```

> **Self-contained:** notebooks don't share memory, so Part 0 rebuilds the small pieces of the assistant we need. Run cells top to bottom. You need the same `AI/.env` with `GROQ_API_KEY` from notebook 11.

## Part 0 — Setup (rebuild the assistant we're grading)

This cell is a condensed copy of the pieces from notebook `11`: the embedding model, a vector store that also returns chunk **indices** (we need those for retrieval metrics), the handbook chunks, and a `rag_answer` function.

If you already ran notebook `11`, the libraries and model are installed, so this just runs.

In [1]:
import os
import json
import numpy as np
from dotenv import load_dotenv
from groq import Groq
from sentence_transformers import SentenceTransformer

load_dotenv()
client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "llama-3.3-70b-versatile"
embedder = SentenceTransformer("all-MiniLM-L6-v2")


def embed(texts):
    return embedder.encode(texts)


def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


HANDBOOK = """
Annual Leave: Full-time employees receive 20 paid annual leave days per year.
Unused leave can be carried forward, up to a maximum of 5 days into the next year.

Work From Home: Employees may work from home up to 3 days per week.
Remote work must be agreed with your manager and logged in the HR portal.

Sick Leave: Employees get 10 paid sick days per year. A doctor's note is required
for any absence longer than 3 consecutive days.

Probation Period: New employees have a probation period of 6 months.
During probation, the notice period for resignation is 2 weeks.

Notice Period: After probation, the standard notice period for resignation is 30 days.

Reimbursements: Work-related expenses are reimbursed within 14 business days
after submitting receipts through the finance portal. The monthly limit is 15000.

Health Insurance: The company provides health insurance covering the employee,
a spouse, and up to two children. Coverage starts from the first working day.
"""

CHUNKS = [p.strip() for p in HANDBOOK.strip().split("\n\n") if p.strip()]
CHUNK_VECTORS = [embed(c) for c in CHUNKS]


def retrieve(question, top_k=3):
    """Return the top-k chunks as (index, score, text), highest score first."""
    q_vec = embed(question)
    scored = [(i, cosine_similarity(q_vec, vec), CHUNKS[i]) for i, vec in enumerate(CHUNK_VECTORS)]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


RAG_SYSTEM = (
    "You are a company HR assistant. Answer using ONLY the context provided. "
    "If the answer is not in the context, say 'I don't know based on the handbook.' Be concise."
)


def rag_answer(question, top_k=3):
    """Return (answer_text, retrieved_indices, context)."""
    hits = retrieve(question, top_k=top_k)
    indices = [i for i, _, _ in hits]
    context = "\n\n".join(text for _, _, text in hits)
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": RAG_SYSTEM},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
    )
    return response.choices[0].message.content, indices, context


for i, c in enumerate(CHUNKS):
    print(f"[{i}] {c[:55]}...")
print("\nSetup complete.")

c:\Users\Mitrah154\AppData\Local\Programs\Python\Python312\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
c:\Users\Mitrah154\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


[0] Annual Leave: Full-time employees receive 20 paid annua...
[1] Work From Home: Employees may work from home up to 3 da...
[2] Sick Leave: Employees get 10 paid sick days per year. A...
[3] Probation Period: New employees have a probation period...
[4] Notice Period: After probation, the standard notice per...
[5] Reimbursements: Work-related expenses are reimbursed wi...
[6] Health Insurance: The company provides health insurance...

Setup complete.


## Part 1 — Ground Truth  *(notebook 08)*

Ground truth = the **known-correct answer** you measure against. It's the `expected` in `assertEqual(actual, expected)`.

We hand-write a small **evaluation dataset**: each item has

- `question` — the input we send to the assistant,
- `expected_keywords` — fact(s) a correct answer must contain (for deterministic scoring),
- `relevant_chunks` — which handbook chunk index actually holds the answer (for retrieval scoring).

```
No ground truth -> no objective way to say "this version is better."
```

Building this is tedious — which is exactly why beginners skip it and their systems quietly degrade.

In [2]:
# Chunk index reference (from Part 0):
#   0 Annual Leave   1 Work From Home   2 Sick Leave
#   3 Probation      4 Notice Period    5 Reimbursements   6 Health Insurance

EVAL_DATASET = [
    {"question": "How many annual leave days do employees get?",
     "expected_keywords": ["20"], "relevant_chunks": [0]},
    {"question": "How many unused leave days can I carry forward?",
     "expected_keywords": ["5"], "relevant_chunks": [0]},
    {"question": "How many days per week can I work from home?",
     "expected_keywords": ["3"], "relevant_chunks": [1]},
    {"question": "How many paid sick days do employees get per year?",
     "expected_keywords": ["10"], "relevant_chunks": [2]},
    {"question": "What is the notice period after probation?",
     "expected_keywords": ["30"], "relevant_chunks": [4]},
    {"question": "What is the monthly reimbursement limit?",
     "expected_keywords": ["15000", "15,000"], "relevant_chunks": [5]},
]

print(f"Eval dataset has {len(EVAL_DATASET)} labelled cases.")
print("Example case:", EVAL_DATASET[0])

Eval dataset has 6 labelled cases.
Example case: {'question': 'How many annual leave days do employees get?', 'expected_keywords': ['20'], 'relevant_chunks': [0]}


## Part 2 — The Simplest Eval: Deterministic Checks  *(notebook 08)*

When the correct answer contains a fixed fact, evaluation is just an **assertion** — the same thing you do in unit tests.

Our assistant returns free text (e.g. *"Employees receive 20 annual leave days."*), so pure `==` exact match is too strict. The practical deterministic check here is **"does the answer contain the expected fact?"**

```
expected_keyword "20"  in  answer  ->  PASS
```

This is free, fast, and 100% objective. **Use it whenever the task allows it** — only reach for the fuzzy LLM-judge metrics (Parts 4-5) when answers are open-ended.

In [42]:
def keyword_match(answer, expected_keywords):
    """PASS if the answer contains ANY of the expected keywords."""
    text = answer.lower()
    return any(kw.lower() in text for kw in expected_keywords)


passed = 0
for case in EVAL_DATASET:
    answer, _, _ = rag_answer(case["question"])
    ok = keyword_match(answer, case["expected_keywords"])
    passed += ok
    print(f"[{'PASS' if ok else 'FAIL'}] {case['question']}")
    print(f"        expected ~ {case['expected_keywords']}  |  answer: {answer[:70]}")

accuracy = passed / len(EVAL_DATASET)
print(f"\nDeterministic accuracy: {passed}/{len(EVAL_DATASET)} = {accuracy:.0%}")

[PASS] How many annual leave days do employees get?
        expected ~ ['20']  |  answer: 20 paid annual leave days per year.
[PASS] How many unused leave days can I carry forward?
        expected ~ ['5']  |  answer: You can carry forward up to a maximum of 5 unused annual leave days in
[PASS] How many days per week can I work from home?
        expected ~ ['3']  |  answer: Up to 3 days per week, with manager agreement and logged in the HR por
[PASS] How many paid sick days do employees get per year?
        expected ~ ['10']  |  answer: 10 paid sick days per year.
[PASS] What is the notice period after probation?
        expected ~ ['30']  |  answer: The standard notice period for resignation after probation is 30 days.
[PASS] What is the monthly reimbursement limit?
        expected ~ ['15000', '15,000']  |  answer: The monthly reimbursement limit is 15000.

Deterministic accuracy: 6/6 = 100%


## Part 3 — Precision, Recall & F1 (retrieval quality)  *(notebook 08)*

Before blaming the LLM for a bad answer, check the **retrieval**. If the relevant chunk was never fetched, the LLM literally cannot answer correctly.

```
Precision = of the chunks we RETURNED, how many were relevant?
Recall    = of all relevant chunks that EXIST, how many did we find?
F1        = one number balancing the two (harmonic mean)
```

We labelled `relevant_chunks` in Part 1, and `retrieve()` tells us which indices were returned — so we can compute these directly. Watch how **precision drops as `top_k` grows** (more chunks returned = more noise), while **recall can only stay equal or rise**.

In [3]:
def precision_recall_f1(retrieved, relevant):
    retrieved_set, relevant_set = set(retrieved), set(relevant)
    hits = len(retrieved_set & relevant_set)
    precision = hits / len(retrieved_set) if retrieved_set else 0.0
    recall = hits / len(relevant_set) if relevant_set else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return precision, recall, f1


def evaluate_retrieval(top_k):
    ps, rs, fs = [], [], []
    for case in EVAL_DATASET:
        retrieved = [i for i, _, _ in retrieve(case["question"], top_k=top_k)]
        p, r, f = precision_recall_f1(retrieved, case["relevant_chunks"])
        ps.append(p); rs.append(r); fs.append(f)
    return np.mean(ps), np.mean(rs), np.mean(fs)


print(f"{'top_k':>6} | {'precision':>9} | {'recall':>7} | {'F1':>5}")
print("-" * 36)
for k in (1, 2, 3, 5):
    p, r, f = evaluate_retrieval(k)
    print(f"{k:>6} | {p:>9.2f} | {r:>7.2f} | {f:>5.2f}")

# Read the table: as top_k rises, recall climbs (we catch the relevant chunk)
# but precision falls (we also drag in irrelevant chunks). That trade-off is
# the core tension of retrieval evaluation.

 top_k | precision |  recall |    F1
------------------------------------
     1 |      1.00 |    1.00 |  1.00
     2 |      0.50 |    1.00 |  0.67
     3 |      0.33 |    1.00 |  0.50
     5 |      0.20 |    1.00 |  0.33


## Part 4 — LLM-as-a-Judge: Relevance  *(notebook 08)*

Keyword matching can't tell if an answer *truly addresses* the question. That needs judgement — and checking thousands of answers by hand doesn't scale. So we use a **second LLM as a grader**.

```
Question + Answer  ->  Judge LLM (given a rubric)  ->  score + reason
```

**Relevance** asks: *does the answer address what was asked?* (An answer can be on-topic but never actually answer the question — low relevance.)

Notice this reuses earlier phases: a **structured prompt** (notebook 05) that returns a **structured output** (notebook 06). We force the judge to reply as JSON `{score, reason}`.

In [4]:
def judge_relevance(question, answer):
    """Use an LLM to score how well the answer addresses the question (1-5)."""
    prompt = f"""You are a strict evaluator.

Question:
{question}

Answer:
{answer}

Score RELEVANCE from 1 to 5:
5 = directly and fully answers the question
1 = does not address the question at all

Respond ONLY as JSON: {{"score": <1-5>, "reason": "<short explanation>"}}"""
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}],
    )
    return json.loads(response.choices[0].message.content)


# A good answer vs a "relevant but unhelpful" drift (example from notebook 08).
q = "How do I reset my password?"
good = "Go to Settings > Security > Reset Password, then check your email for a link."
drift = "Passwords are important for security and should be changed regularly."

print("GOOD :", judge_relevance(q, good))
print("DRIFT:", judge_relevance(q, drift))

GOOD : {'score': 5, 'reason': 'Directly provides step-by-step instructions to reset password'}
DRIFT: {'score': 1, 'reason': 'The answer does not provide instructions on how to reset a password.'}


## Part 5 — LLM-as-a-Judge: Faithfulness (anti-hallucination)  *(notebook 08)*

This is the most important metric for RAG.

```
Faithfulness = Is the answer SUPPORTED by the retrieved context?
```

Don't confuse it with relevance:

```
Relevance     -> is the answer ABOUT the question?
Faithfulness  -> is the answer SUPPORTED by the source?
```

The most dangerous failure is an answer that is **relevant but unfaithful** — an on-topic hallucination that *sounds* right but invents facts. The judge gets the **context** and checks every claim against it. Below we feed it a deliberately unfaithful answer to confirm it gets caught.

In [34]:
def judge_faithfulness(context, answer):
    """Score whether every claim in the answer is supported by the context (1-5)."""
    prompt = f"""You are a strict evaluator checking for hallucination.

Context (the only allowed source of truth):
{context}

Answer:
{answer}

Score FAITHFULNESS from 1 to 5:
5 = every claim is supported by the context
1 = the answer invents facts not in the context

Respond ONLY as JSON: {{"score": <1-5>, "reason": "<short explanation>"}}"""
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}],
    )
    return json.loads(response.choices[0].message.content)


context = "Employees receive 20 annual leave days. Unused leave can be carried forward up to 5 days."
faithful = "You receive 20 annual leave days, and up to 5 can be carried forward."
unfaithful = "You receive 20 annual leave days and can carry forward up to 15 days, plus a bonus week."

print("FAITHFUL  :", judge_faithfulness(context, faithful))
print("UNFAITHFUL:", judge_faithfulness(context, unfaithful))

FAITHFUL  : {'score': 5, 'reason': 'The answer directly reflects the information given in the context.'}
UNFAITHFUL: {'score': 1, 'reason': 'invents facts: carry forward up to 15 days and bonus week'}


## Part 6 — LLM Outputs Aren't Deterministic  *(notebook 08)*

Run the **same prompt twice** and the wording can differ — that's the sampling (temperature / top-p) from notebook `01`. This breaks naive exact-match eval and makes scores wobble between runs.

How we handle it:

```
* Lower temperature (-> 0) for repeatable, gradeable outputs
* OR run each case several times and average the score
* Use meaning-based metrics (relevance/faithfulness/judge) for open text
```

Below: ask the same question with `temperature=1.0` a few times and see the wording change even though the fact stays the same. Then note that our eval uses `temperature=0` exactly to keep things stable.

In [40]:
question = "How many annual leave days do employees get?"
context = CHUNKS[0]

print("Same question, temperature=1.0, three runs:\n")
for run in range(3):
    response = client.chat.completions.create(
        model=MODEL,
        temperature=2.0,
        messages=[
            {"role": "system", "content": RAG_SYSTEM},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
    )
    print(f"run {run + 1}: {response.choices[0].message.content}")

print("\nDifferent wording, same fact ('20'). This is why open-ended answers")
print("need meaning-based metrics, and why our eval pins temperature=0.")

Same question, temperature=1.0, three runs:

run 1: 20 paid annual leave days per year.
run 2: 20 paid annual leave days per year.
run 3: 20 paid annual leave days per year.

Different wording, same fact ('20'). This is why open-ended answers
need meaning-based metrics, and why our eval pins temperature=0.


## Part 7 — The Full Eval Pipeline  *(notebook 08)*

Now we combine everything into one run over the whole dataset. For each case we compute:

- **retrieval**: precision / recall / F1 (Part 3)
- **answer correctness**: keyword match (Part 2)
- **relevance** and **faithfulness**: LLM judge (Parts 4-5)

And because quality isn't the only axis, we also track:

```
Quality  -> is it right?
Latency  -> how fast? (seconds per answer)
Cost     -> how many tokens? (tokens = money, notebook 01)
```

The output is a per-case table plus aggregate scores — your "test report" for the assistant.

In [43]:
import time


def answer_with_metrics(question, top_k=3):
    """Run RAG and capture answer, retrieved indices, context, tokens, and latency."""
    hits = retrieve(question, top_k=top_k)
    indices = [i for i, _, _ in hits]
    context = "\n\n".join(text for _, _, text in hits)

    start = time.time()
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": RAG_SYSTEM},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
    )
    latency = time.time() - start
    answer = response.choices[0].message.content
    return answer, indices, context, response.usage.total_tokens, latency


def run_eval(dataset, top_k=3):
    rows = []
    for case in dataset:
        answer, indices, context, tokens, latency = answer_with_metrics(case["question"], top_k)
        p, r, f = precision_recall_f1(indices, case["relevant_chunks"])
        rows.append({
            "question": case["question"],
            "correct": keyword_match(answer, case["expected_keywords"]),
            "precision": p, "recall": r, "f1": f,
            "relevance": judge_relevance(case["question"], answer)["score"],
            "faithfulness": judge_faithfulness(context, answer)["score"],
            "tokens": tokens, "latency": latency,
        })
    return rows


def summarize(rows):
    n = len(rows)
    print(f"{'correctness':>12}: {sum(r['correct'] for r in rows) / n:.0%}")
    print(f"{'recall':>12}: {np.mean([r['recall'] for r in rows]):.2f}")
    print(f"{'F1':>12}: {np.mean([r['f1'] for r in rows]):.2f}")
    print(f"{'relevance':>12}: {np.mean([r['relevance'] for r in rows]):.2f} / 5")
    print(f"{'faithfulness':>12}: {np.mean([r['faithfulness'] for r in rows]):.2f} / 5")
    print(f"{'avg latency':>12}: {np.mean([r['latency'] for r in rows]):.2f} s")
    print(f"{'avg tokens':>12}: {np.mean([r['tokens'] for r in rows]):.0f}")


print("Running full evaluation (this calls the LLM several times)...\n")
results = run_eval(EVAL_DATASET, top_k=3)

for r in results:
    print(f"[{'OK ' if r['correct'] else 'BAD'}] rel={r['relevance']} faith={r['faithfulness']} "
          f"recall={r['recall']:.1f}  {r['question'][:45]}")

print("\n=== AGGREGATE SCORES (top_k=3) ===")
summarize(results)

Running full evaluation (this calls the LLM several times)...

[OK ] rel=5 faith=5 recall=1.0  How many annual leave days do employees get?
[OK ] rel=5 faith=5 recall=1.0  How many unused leave days can I carry forwar
[OK ] rel=5 faith=5 recall=1.0  How many days per week can I work from home?
[OK ] rel=5 faith=5 recall=1.0  How many paid sick days do employees get per 
[OK ] rel=5 faith=5 recall=1.0  What is the notice period after probation?
[OK ] rel=5 faith=5 recall=1.0  What is the monthly reimbursement limit?

=== AGGREGATE SCORES (top_k=3) ===
 correctness: 100%
      recall: 1.00
          F1: 0.50
   relevance: 5.00 / 5
faithfulness: 5.00 / 5
 avg latency: 0.21 s
  avg tokens: 190


## Part 8 — A/B Comparison: prove a change is better  *(notebook 08)*

This is the whole point of evaluation. We change **one thing** — `top_k` (1 vs 3) — and re-run the *same* eval. Now "which is better?" is a number, not an opinion.

```
Change ONE thing -> re-run evals -> keep it only if scores improve.
```

Expect `top_k=1` to sometimes miss the relevant chunk (lower recall -> lower correctness/faithfulness), while `top_k=3` retrieves more reliably at the cost of a few more tokens. That cost/quality trade-off is exactly the decision real teams make.

In [44]:
print("=== Version A: top_k = 1 ===")
results_a = run_eval(EVAL_DATASET, top_k=1)
summarize(results_a)

print("\n=== Version B: top_k = 3 ===")
results_b = run_eval(EVAL_DATASET, top_k=3)
summarize(results_b)


def score(rows):
    return {
        "correctness": np.mean([r["correct"] for r in rows]),
        "recall": np.mean([r["recall"] for r in rows]),
        "faithfulness": np.mean([r["faithfulness"] for r in rows]),
        "avg_tokens": np.mean([r["tokens"] for r in rows]),
    }


a, b = score(results_a), score(results_b)
print("\n=== VERDICT ===")
for metric in a:
    arrow = "B better" if b[metric] > a[metric] else ("A better" if a[metric] > b[metric] else "tie")
    if metric == "avg_tokens":  # for cost, lower is better
        arrow = "A cheaper" if a[metric] < b[metric] else ("B cheaper" if b[metric] < a[metric] else "tie")
    print(f"{metric:>13}: A={a[metric]:.2f}  B={b[metric]:.2f}  -> {arrow}")

=== Version A: top_k = 1 ===
 correctness: 100%
      recall: 1.00
          F1: 1.00
   relevance: 5.00 / 5
faithfulness: 5.00 / 5
 avg latency: 0.23 s
  avg tokens: 127

=== Version B: top_k = 3 ===
 correctness: 100%
      recall: 1.00
          F1: 0.50
   relevance: 5.00 / 5
faithfulness: 5.00 / 5
 avg latency: 0.91 s
  avg tokens: 188

=== VERDICT ===
  correctness: A=1.00  B=1.00  -> tie
       recall: A=1.00  B=1.00  -> tie
 faithfulness: A=5.00  B=5.00  -> tie
   avg_tokens: A=127.00  B=188.50  -> A cheaper


## Recap — what you just built

```
Eval dataset (questions + ground truth)
        |
        v
Run each question through the assistant
        |
        v
Score:  correctness (keywords)          <- deterministic
        precision / recall / F1         <- retrieval
        relevance + faithfulness        <- LLM judge
        latency + tokens                <- speed + cost
        |
        v
Aggregate -> compare versions -> ship the better one
```

Every concept from notebook `08` is now runnable:

- **Ground truth** — a labelled eval dataset.
- **Exact / deterministic checks** — keyword/assertion scoring.
- **Precision / Recall / F1** — retrieval quality, and the top_k trade-off.
- **Relevance** — does the answer address the question? (LLM judge)
- **Faithfulness** — is it supported by the context? (anti-hallucination judge)
- **Non-determinism** — why scores wobble, and using temperature=0.
- **Pipeline + latency/cost** — quality is not the only axis.
- **A/B comparison** — change one thing, prove it with numbers.

### Exercises

1. **Grow the dataset.** Add 4 more cases (including a "not in handbook" question whose expected answer is "I don't know"). Re-run Part 7.
2. **Catch a regression.** Weaken `RAG_SYSTEM` (remove "use ONLY the context"), re-run, and watch faithfulness drop.
3. **Judge the judge.** Hand-label relevance for 3 answers, then compare your scores to the LLM judge's. Do they agree?
4. **New axis.** Add an `expected_source` to each case and score whether the *correct chunk index* was the top-1 retrieved (a stricter retrieval metric).
5. **Model A/B.** Swap `MODEL` to a smaller Groq model for Version A and compare quality vs latency vs tokens.
6. **Cost report.** Convert `avg_tokens` into an estimated dollar cost using a price-per-token and print cost per 1,000 requests.

> With this, "improving" the assistant stops being vibes and becomes engineering. Next up (notebook `09` Agents) you'll let the assistant *act* on its own — which makes evaluation even more essential.